## Node-style hooks函数用法
支持两种用法
- 装饰器是函数式挂载，把一个hook快速挂载到Agent的某个节点。
- 类写法是对象化中间件，把中间件封装为一个可配置、可复用、可扩展的组件。

In [2]:
# 基于装饰器实现
# 1、模型的初始化
import os
from dotenv import load_dotenv
from langchain_qwq import ChatQwen

custom_profile = {
"max_input_tokens": 128_000 # 最大上下文长度
}

# 从.env文件中加载环境变量
load_dotenv(override=True)
# 模型的初始化
model = ChatQwen(
    model="qwen3.6-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    profile=custom_profile, # 手动添加的配置项
)

In [ ]:
from langchain.agents.middleware import (
    before_model,
    after_model,
    before_agent,
    after_agent,
    AgentState,
    AgentMiddleware,
)
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


# 1. 定义 before_model 钩子
@before_model
def before_model_middleware(
    state: AgentState,  # 当前运行的状态，包含了用户的消息列表等信息
    runtime: Runtime # 是这次运行时的上下文和基础设施。
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_model <- "
    return None


# 2. 定义 after_model 钩子
@after_model
def after_model_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> after_model <- "
    return None


# 3. 定义 before_agent 钩子
@before_agent
def before_agent_middleware(
    state: AgentState, runtime: Runtime
) -> dict[str, Any] | None:
    state["messages"][-1].content += " -> before_agent <- "
    return None


# 4. 定义 after_agent 钩子
@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    state["messages"][-1].content += " -> after_agent <- "
    return None


agent = create_agent(
    model=model,
    middleware=[
        before_model_middleware,
        after_model_middleware,
        before_agent_middleware,
        after_agent_middleware,
    ],  # 👈 添加中间件
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:

    msg.pretty_print()

================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好！已收到你的输入流标识：`你好啊 -> before_agent <-  -> before_model <-`。

在日常交互中，这通常表示数据会先经过 `before_agent`（如权限校验、路由、日志记录等），再进入 `before_model`（如 Prompt 组装、上下文裁剪、安全过滤等），最后才由大模型生成回复。

如果你是在调试自定义 Hook、中间件或工作流框架，我可以：
- 帮你梳理各阶段的输入/输出格式
- 提供对应代码片段（Python/JS/Node.js 等）
- 模拟后续 `model_output -> after_model <-` 的完整流程

需要我侧重哪一部分？随时告诉我～ 😊 -> after_model <-  -> after_agent <-


# 基于类实现
1. 必须继承 AgentMiddleware ← 这个固定
2. 方法名固定 ( before_model , after_model ) ← 这个固定
3. 类名随意 ← 这个不固定

LangGraph 只看：

是否继承 AgentMiddleware？

是否有 before_model / after_model 等方法？

In [3]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from typing import Any


class MyMiddleware(AgentMiddleware):
    def __init__(self):
        super().__init__()

    def before_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        print("before_model")
        print(state)
        print(runtime)
        state["messages"][-1].content += " -> before_model <- "
        return None

    def after_model(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> after_model <- "
        return None

    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        state["messages"][-1].content += " -> before_agent <- "
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        state["messages"][-1].content += " -> after_agent <- "
        return None


my_middleware = MyMiddleware()

agent = create_agent(
    model=model,
    middleware=[my_middleware],
)

response = agent.invoke({
    "messages": [HumanMessage("你好啊")],
})

for msg in response["messages"]:
    msg.pretty_print()

before_model
{'messages': [HumanMessage(content='你好啊 -> before_agent <- ', additional_kwargs={}, response_metadata={}, id='110a3421-4254-463b-9c13-338257781aa3')]}
Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x118afeac0>, heartbeat=<function _no_op_heartbeat at 0x108c8a5c0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f1794fd-9a61-6cd4-8001-8afa611864cd', checkpoint_ns='MyMiddleware.before_model:b7b86670-38da-59a3-584b-4284c08d3fc4', task_id='b7b86670-38da-59a3-584b-4284c08d3fc4', thread_id=None, run_id=None, node_attempt=1, node_first_attempt_time=1783351928.611798), server_info=None, control=<langgraph.runtime.RunControl object at 0x10ff373a0>)
================================ Human Message =================================

你好啊 -> before_agent <-  -> before_model <- 
================================== Ai Message ==================================

你好呀！👋  
你输入的格式里带了一些类似框架运行时的标记（`before_agent` / `before_model`），不过这不

1. before_model通用场景：
- 消息修剪（trim messages）
- PII 脱敏
- 输入验证
- 条件路由

2.  after_model 通常的场景：
- 输出验证
- 格式化响应
- 统计信息
- 状态更新

### 2种方法的统一
装饰器底层会基于我们重写的方法构造一个AgentMiddleware子类的实例，以@after_model为例：

In [ ]:
# 这是after_model最终返回的内容
return type(
    middleware_name,
    (AgentMiddleware,),
    {
        "state_schema": state_schema or AgentState,
        "tools": tools or [],
        "after_model": wrapped,
},
)()

这是after_model最终返回的内容。

上述代码中的wrapped是after_model内部的装饰器，代码如下


In [ ]:
def wrapped(
    _self: AgentMiddleware[StateT, ContextT],
    state: StateT,
    runtime: Runtime[ContextT],
) -> dict[str, Any] | Command[Any] | None:
    return func(state, runtime) # type: ignore[return-value]

In [ ]:
return type(
    middleware_name,
    (AgentMiddleware,),
    {
        "state_schema": state_schema or AgentState,
        "tools": tools or [],
        "after_model": func(state, runtime),
    },
)()

而 func(state, runtime) 正是我们定义的、被 @after_model 修饰的函数，在上述案例中对应的是
after_model_middleware，上述代码的含义是

    1. 创建一个AgentMiddleware的子类
   
    2. 类名为middleware_name，即创建agent时传递的中间件名称，上述案例中after_model_middleware

    3. 这个子类有两个属性 state_schema 和 tools

    4. 有一个方法： after_model ，逻辑等同于 func(state, runtime) 。

    5. 最后的括号 () 表示实例化子类，返回一个对象

#### 参数说明
- state: 是一个AgentState实例，维护Agent运行中的状态，这类状态会随着Agent的运行而发生变化，包括消息列表。

- runtime: 是一个Runtime实例，维护Agent运行过程中的上下文环境，包括上下文、长期记忆等。

#### 返回值说明
- 返回None:不修改状态

- 返回字典：更新状态

- 返回{“jump_to”:...}：控制流程
```python
def before_model(self, state, runtime):
    if state.get("count", 0) > 10:
        return {"jump_to": "__end__"} # 跳过模型，直接结束
    return None
```
  - "__end__" - 结束 Agent
  - "tools" - 跳到工具节点
  - 其他自定义节点

#### 装饰器参数：can_jump_to
涉及到Node-style的四个hook函数可以接收额外参数 `can_jump_to` 。
钩子函数可以 `改变Agent正常的运行轨迹` 。比如：发现上下文窗口溢出，直接跳转至结尾，提前终止整
个Agent。
can_jump_to 决定了钩子函数可以直接跳转至流程的哪些位置，可取值如下：
- end：跳转至Agent流程末尾，或第一个after_agent钩子，直接终止整个流程。
- tools：跳转至工具节点。
- model：跳转至模型节点，或第一个before_model钩子。